# SODA: from instrument files to one link you can share

This notebook turns the files your instrument (or analysis pipeline) wrote into **one folder,
`export/`**, that you upload to the cloud as it is, and **one link** that opens all your
datasets in the browser. The whole job:

1. **Install once** (Section 0 below), then open this notebook with SODA's `.venv`.
2. **Put each dataset in its own folder** under `inputs/` in your work folder (Section 1).
3. **Set `WORK`**, your work folder, in the *Settings* cell (and `CLOUD_URL`, your cloud
   folder's link, once you have it).
4. **Run all cells** (*Run > Run All Cells*). Section 2 says what each folder is: fix any
   `FIX` line and run again. Every step skips what is already done, so you can stop and start
   again. The run stops at Section 7 with a question: that is expected.
5. **In Section 7, answer "is this data public?"**, upload `export/` as it is, run the link
   test, and share the link it prints.

Sections: 0 Setup, 1 Put your files here, 2 Check the inputs, 3 Prepare, 4 Export,
5 De-identification check, 6 Look before you upload, 7 Upload and share. The code behind each
cell is in `notebooks/soda_notebook.py`; the same steps as commands are in
[docs/PREPARING_DATA.md](../docs/PREPARING_DATA.md).

## 0. Setup

Once, from the repository folder (keep the path short on Windows), with Python 3.12 or 3.13.
SODA gets its own environment because it needs older `zarr` and `tifffile` than many tools:

```
powershell -ExecutionPolicy Bypass -File install.ps1    (macOS / Linux: bash install.sh)
.venv\Scripts\activate              (macOS / Linux: source .venv/bin/activate)
pip install jupyter
```

The install ends with **SODA is installed** (it is safe to run again). Then open this notebook
with that environment's Jupyter, or pick `.venv` as the kernel in VS Code. The next cell must
show no `MISSING` core package and no `WRONG VERSION` line.
By hand instead: `pip install -r web/SpatialViewer/requirements.txt -r web/SpatialViewer/requirements-clustering.txt`
(the second file is optional: without it the export skips the whole-section UMAP and clusters).

In [ ]:
import sys
from pathlib import Path
# this notebook's code, notebooks/soda_notebook.py, from Jupyter's working folder (notebooks/ or the repository)
here = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "notebooks" / "soda_notebook.py").exists()), None)
if here is None:
    raise FileNotFoundError("Open this notebook from inside the SODA repository (the folder that has notebooks/soda_notebook.py).")
sys.path.insert(0, str(here / "notebooks"))
import soda_notebook as soda
soda.setup()

### Settings: edit this cell

- **`WORK`** is one folder on a big, fast disk. `inputs/` (your files), `cache/` (working
  copies, large, never uploaded) and `export/` (what you upload) go inside it. Plan for about
  2-3 times the size of your raw files.
- **`CLOUD_URL`** is the link to the cloud folder the data will go to (on Azure a blob
  container, e.g. `https://myaccount.blob.core.windows.net/spatial`). Paste only the address,
  never a key or a `?sv=...` pass. It can stay empty until Section 7.

In [ ]:
WORK = r"D:\SODA_work"   # EDIT: a big disk; inputs/, cache/ and export/ go inside
CLOUD_URL = ""           # EDIT before Section 7: your cloud folder's link, e.g. https://myaccount.blob.core.windows.net/spatial
soda.set_work(WORK, CLOUD_URL)

### Advanced settings (optional)

**You do not need to change anything here.** Run the cell anyway: it sets the defaults the
rest of the notebook uses. Every setting is explained in
[docs/ADVANCED.md, section 4](../docs/ADVANCED.md#4-every-notebook-setting); the key file in
[docs/DEIDENTIFICATION.md](../docs/DEIDENTIFICATION.md); the barcode / cell id lists
(`INCLUDE_IDS`) in [docs/ADVANCED.md](../docs/ADVANCED.md#barcode-and-cell-id-lists).

In [ ]:
soda.advanced(
    SODA_PAGE = "https://soda.mgifive.org/",  # the page your link opens: the shared SODA page (your own copy of SODA: put its address)
    KEY_FILE = soda.DEFAULT_KEY_FILE,    # your de-identification key, if any ($SPATIALVIZ_KEY, else ~/private/soda/sample_id_key.csv): keep it outside WORK, never upload it
    ALWAYS_CHECK_IDS = False,            # True: Section 5 checks for sample IDs even without a key file
    TITLE = "",                          # a name for the whole collection, e.g. "Our nerve atlas" ("" keeps the one already set)
    READ_PASS = "",                      # private folder only: a READ-ONLY pass (Azure SAS, starts with sv=); anyone with the link can read the data until it expires
    TILE_FORMAT = "jpg",                 # "jpg" is several times smaller; "png" is lossless (use it when exact intensities matter)
    VISIUM_BINS = "008um,016um",         # Visium HD layers to keep; the 2 um layer (002um) is huge and mostly one transcript per bin
    WORKERS = 8,                         # parallel tile writers during the export
    MARKERS = {"visium": "", "proteomics": ""},  # marker lists for suggested cluster names, e.g. str(soda.SV / "markers" / "nerve.json"); "" = the generic list
    REBUILD = False,                     # True: prepare datasets again even when cache/<id>/ has them (then delete export/<id>/ too)
    INCLUDE_IDS = True,                  # barcode / cell id lists in the export (checked for sample IDs); False: none (--no-ids)
)

## 1. Put your files here

**Put each dataset in its own folder under `inputs/`. Any folder name works, and you do not
have to say what kind of data it is:** Section 2 looks at the files and works it out.

```
inputs/
  nerve_section1/      <- for example a Space Ranger outs/ folder + the microscope image
  tumour_xenium/       <- a Xenium output folder, exactly as the instrument wrote it
  slide_07.qptiff      <- one image file (QPTIFF, OME-TIFF) or one .h5ad can sit here on its own
```

The folder name becomes the dataset id, so **do not put patient or sample identifiers in it**.
A name starting with `_` or `.` is left out. What each instrument's folder must hold, zipped
output, big files kept where they are, and your own UMAP and clusters:
[docs/PREPARING_DATA.md, Using the notebook](../docs/PREPARING_DATA.md#using-the-notebook).

In [ ]:
soda.list_inputs()

## 2. Check the inputs

The next cell looks at every folder under `inputs/` (names, sizes and file headers only;
nothing is converted yet) and shows one table per dataset: what it is, **how sure** that is,
what was found and what is missing. It is the same check as `prepare.py detect` and
`prepare.py batch`.

- **certain**: nothing to do.
- **NOT SURE: pick one**: the table prints the line to copy into `OPTIONS`. It never guesses.
- **recognised, but something is missing**: the table says which file.
- **not recognised**: say what it is with `"kind"`, or check the folder against Section 1.

**`OPTIONS` is the one place to change anything about a dataset**, keyed by dataset id (the
folder's name). Leave a dataset out to use what was detected. Every key (`kind`, `image`,
`label`, `pixel_size`, `crop`, `user_layer` ...):
[docs/PREPARING_DATA.md, Settings per dataset](../docs/PREPARING_DATA.md#settings-per-dataset-options).

In [ ]:
OPTIONS = {
    # One line per dataset (the folder's name under inputs/), only where you want to change something.
    # "my_folder":      {"kind": "xenium"},                # the table says "pick one", or got the kind wrong
    # "my_visium":      {"kind": "visium", "image": r"E:\scans\he_fullres.tif"},   # the image given to spaceranger
    # "nerve_section1": {"label": "Normal nerve"},
    # "neuroma_panel":  {"label": "Painful neuroma", "pixel_size": 0.5},
    # "big_scan":       {"src": r"E:\scans\slide_07.qptiff", "label": "Whole slide"},
    # "hd_section2":    {"user_layer": "008um"},   # + my_umap.csv / my_clusters.csv in its folder
    # "codex_slide":    {"cells": r"E:\seg\codex_slide_cp_masks.tif"},   # your own segmentation (label mask or GeoJSON)
}
soda.check_inputs(OPTIONS)

Optional: look inside a proteomics or H&E file before building (axes, size, levels, channel
names). Nothing is written.

In [ ]:
soda.inspect_images()

## 3. Prepare: read each dataset once into `cache/`

The slow step. The first cell checks the memory (it only warns). The second reads each dataset
once into a multi-resolution copy plus a gene-by-bin index. The cache holds the full-resolution
raw image, so **it stays on this computer and is never uploaded**. A dataset already in
`cache/` is skipped; `REBUILD = True` in the advanced settings redoes it (then also delete
`export/<id>/`, or old tiles stay). Single-cell RNA-seq has no cache: it goes straight to
`export/`.

In [ ]:
soda.memory_check()

In [ ]:
soda.prepare_all()

## 4. Export: write the upload-ready files into `export/`

Each prepared dataset gets `export/<id>/`: the image tiles, the gene (or per-16 µm-square
protein) data with a UMAP, the barcode / cell id lists (by default; checked for sample IDs)
and `manifest.json`, without your local file paths. Then **`export/datasets.json`** lists every
dataset with the name and heading the page shows; you may rename or regroup them in it with
any text editor, and running this cell again keeps your edits
([docs/SHARE_YOUR_DATA.md](../docs/SHARE_YOUR_DATA.md#what-you-can-change-in-datasetsjson)).
What each file is: [docs/PREPARING_DATA.md, Export](../docs/PREPARING_DATA.md#export).

Re-running skips what already exists. Check the log for `[export] verified N channel(s)` and
no line starting with `***`.

In [ ]:
soda.export_all()

## 5. De-identification check

Scans every file name and every text value in `export/` (the id lists too) for anything that
looks like a sample ID. **With a key file** (`KEY_FILE` in the advanced settings, or
`ALWAYS_CHECK_IDS = True`) it must end with **`[deid] clean`**, and Section 7 will not continue
until it does. Without one it only warns.

If it lists matches: rename the source (folder name, label, channel or column name), delete
`export/<id>/` (and `cache/<id>/` for a label or channel name: Section 3 skips a dataset already
in `cache/`), and run Sections 3-5 again. A gene name can match by accident, so read the list
before renaming anything. Read the labels and channel names yourself too: the check only knows
the ID patterns in `deid.py` ([docs/DEIDENTIFICATION.md](../docs/DEIDENTIFICATION.md)). You are
responsible for what you publish.

In [ ]:
soda.deid_check()

## 6. Look before you upload

The size of each dataset (many hosts limit file count and total size), then the exact
`export/` folder opened in your browser on this computer: the first link opens the whole
folder the way your shared link will. Nothing leaves your computer here. Click through every
dataset, then run the third cell to stop the servers.

In [ ]:
soda.sizes()

In [ ]:
soda.preview()

In [ ]:
# soda.stop_preview()   # when you are done looking: stops the two local servers (they also stop with the notebook)

## 7. Upload and share

This is the last step: your `export/` folder is ready and Section 5 said it is clean.
**Who may see the data?**

```
Can anyone see this data?
|
+-- Yes, public --> upload export/ to your own cloud storage (public-read)
|                   --> allow https://soda.mgifive.org to read it (CORS, with Range)
|                   --> a link on the shared page, soda.mgifive.org/?data=<your folder>
|                       (the link for a paper), or your own copy of SODA
|                       (Use this template + GitHub Pages)
|
+-- No, private --> Who needs to see it?
                    +-- a few people, for a while: private folder + read pass (SAS),
                    |   sent privately, never in a paper
                    +-- your lab, long term: a password site on Azure (deploy/azure)
                    +-- only you: upload nothing; Section 6's preview or the desktop app
```

GitHub holds only the SODA page, never data. Box, Google Drive and Dropbox cannot serve the
data. The full chart and every step: [docs/SHARE_YOUR_DATA.md](../docs/SHARE_YOUR_DATA.md).

1. **Answer the question at the top of the next cell: is this data public?** The cell stops
   until you do. Fill in `CLOUD_URL` in the *Settings* cell (Section 0) first if it is empty.
2. Run the cell. It prints the upload command and **the link to share**. It runs nothing and
   uploads nothing.
3. Upload **the whole `export/` folder as it is** (one subfolder per dataset, plus
   `datasets.json`). Never upload `cache/`, `inputs/` or the key file.
4. Run the cell after it. It reads your uploaded folder the way a browser would and tells you
   whether the SODA page may read it. If not, it prints the exact setting to change.

What the answer means ([docs/SHARE_YOUR_DATA.md](../docs/SHARE_YOUR_DATA.md),
[docs/HOSTING.md](../docs/HOSTING.md)):

- **`"yes"`, public**: anyone with the link can open (and download) the data. Only for data
  that may be published, such as examples and published data, with nothing about a patient.
  You pay for what people download, so keep public sets small.
- **`"no"`, private**: the data stays in a private folder. Readers either use your own
  password-protected SODA site (`deploy/azure/README.md`; the cell prints its `config.js`
  lines), or get a link that carries a read-only pass (`READ_PASS` in the advanced settings).
  Anyone holding such a link can read the data until the pass expires.

In [ ]:
# ANSWER THIS FIRST. Is this data public?
#   "yes" = anyone with the link may see and download it (published or example data, nothing about a patient)
#   "no"  = private: only people with your site's password or a read pass may see it
IS_THIS_DATA_PUBLIC = ""      # EDIT: "yes" or "no"
soda.share(IS_THIS_DATA_PUBLIC)

### Test the link (after uploading)

A browser lets the SODA page read files from your storage only if the storage allows the
page's address (a CORS rule). This cell asks for your `datasets.json` the way a browser on
the SODA page would, then reads a few bytes of one gene pack with a Range request (that is how
the page reads one gene). If something is missing, it prints the exact fix for your storage.
Set it once, wait a minute, and run this cell again until it says OK.

In [ ]:
soda.test_link()

### Before you share the link

- [ ] Section 5 says `[deid] clean` (or was skipped because you have no key file), and you read the labels and channel names yourself.
- [ ] You answered "is this data public?" with "yes" only for data anyone may see.
- [ ] Every dataset opened and looked right in Section 6.
- [ ] Only `export/` was uploaded: no `cache/`, raw files or key file.
- [ ] The link test says OK, and the link opened every dataset in a private browser window.
- [ ] The names in `datasets.json` (and `config.js`, if you have your own site) name the tissue and condition only.
- [ ] Private data: the password works and a link without it shows nothing; a link with a read pass goes only to people who may see the data.